In [ ]:
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from copy import deepcopy

Hamiltonian upload

In [ ]:
# Load the exact Stage-6 artifact exported by ADMET_future_curation_complete.
from pathlib import Path
import pickle

root_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
SUBMISSION_ROOT = next((p for p in root_candidates if (p / "requirements.txt").is_file()), None)
if SUBMISSION_ROOT is None:
    raise FileNotFoundError("Run inside HORIZON_PROJECTS_FINAL_SUBMISSION or its pipeline folder")
ARTIFACT_DIR = SUBMISSION_ROOT / "artifacts"
CURATED_PATH = ARTIFACT_DIR / "curated_admet_hamiltonians.pkl"
if not CURATED_PATH.is_file():
    raise FileNotFoundError("Run ADMET_future_curation_complete.ipynb first: " + str(CURATED_PATH))
with CURATED_PATH.open("rb") as handle:
    hamiltonians = pickle.load(handle)
required = {"hamiltonian_A_interaction_ising", "hamiltonian_B_diagonal_feature"}
if not required.issubset(hamiltonians):
    raise ValueError("Stage-6 artifact is missing required Hamiltonian families")
print("Loaded:", CURATED_PATH)


Fingeprint upload

Problem hamiltonian

In [ ]:
# LOAD PROBLEM HAMILTONIAN COMPONENTS

hamiltonian_A = hamiltonians["hamiltonian_A_interaction_ising"]
hamiltonian_B = hamiltonians["hamiltonian_B_diagonal_feature"]

print("Datasets:", list(hamiltonian_A.keys()))

print("\nHamiltonian A datasets:", len(hamiltonian_A))
print("Hamiltonian B datasets:", len(hamiltonian_B))

In [ ]:
# BUILD PROBLEM HAMILTONIANS
# H{problem} = H_A + H_B

problem_hamiltonians = {}

for dataset_name in hamiltonian_A:

    interaction_terms = deepcopy(
        hamiltonian_A[dataset_name]["pauli_terms"]
    )

    feature_terms = deepcopy(
        hamiltonian_B[dataset_name]["pauli_terms"]
    )

    problem_hamiltonians[dataset_name] = {
        "hamiltonian_type": "problem_hamiltonian",
        "formula": "H_problem = H_A + H_B",
        "selected_features":
            hamiltonian_A[dataset_name]["selected_features"],
        "X_selected_scaled":
            hamiltonian_B[dataset_name]["X_selected_scaled"],
        "pauli_terms":
            interaction_terms + feature_terms,
    }

    print(
        dataset_name,
        "->",
        len(problem_hamiltonians[dataset_name]["pauli_terms"]),
        "Pauli terms"
    )

In [ ]:
# BUILD ADIABATIC HAMILTONIANS
# H_ad(λ) = (1-λ) H_initial + λ H_problem

adiabatic_hamiltonians = {}

for dataset_name, problem in problem_hamiltonians.items():

    num_features = len(problem["selected_features"])

    # Initial transverse-field Hamiltonian
    initial_terms = []

    for i in range(num_features):
        initial_terms.append({
            "coefficient": -1.0,
            "pauli_string": f"X{i}",
            "locality": 1
        })

    adiabatic_hamiltonians[dataset_name] = {

        "hamiltonian_type": "adiabatic_hamiltonian",

        "formula":
            "H_ad(lambda) = (1-lambda) H_initial + lambda H_problem",

        "selected_features":
            problem["selected_features"],

        "X_selected_scaled":
            problem["X_selected_scaled"],

        "initial_hamiltonian":
            initial_terms,

        "problem_hamiltonian":
            deepcopy(problem["pauli_terms"])
    }

    print(
        dataset_name,
        "->",
        len(initial_terms),
        "initial terms +",
        len(problem["pauli_terms"]),
        "problem terms"
    )

Derivation

In [ ]:
# BUILD O0 = dH_ad / dλ
# O0 = H_problem - H_initial


O0_hamiltonians = {}

for dataset_name, Had in adiabatic_hamiltonians.items():

    O0_terms = []

    # H_problem
    O0_terms.extend(
        deepcopy(Had["problem_hamiltonian"])
    )

    # -H_initial
    for term in Had["initial_hamiltonian"]:

        new_term = deepcopy(term)
        new_term["coefficient"] *= -1

        O0_terms.append(new_term)

    O0_hamiltonians[dataset_name] = {

        "hamiltonian_type": "O0",

        "formula": "O0 = dH_ad/dlambda = H_problem - H_initial",

        "selected_features":
            Had["selected_features"],

        "X_selected_scaled":
            Had["X_selected_scaled"],

        "pauli_terms":
            O0_terms
    }

    print(
        dataset_name,
        "->",
        len(O0_terms),
        "Pauli terms"
    )

In [ ]:
import pennylane as qml
from pennylane.pauli import PauliWord

In [ ]:
from pennylane.pauli import PauliWord


def pauli_string_to_word(pauli_string):

    mapping = {}

    for token in pauli_string.split():

        pauli = token[0]
        wire = int(token[1:])

        mapping[wire] = pauli

    return PauliWord(mapping)

## First order operator 
## Compute O_1=[H_{\text{initial}},H_{\text{problem}}] for one initial term

In [ ]:
# BUILD FIRST-ORDER OPERATOR O1 = [H_initial, H_problem]

import pennylane as qml

O1 = {}

for dataset_name in problem_hamiltonians:

    print(f"Processing {dataset_name}...")

    problem_terms = problem_hamiltonians[dataset_name]["pauli_terms"]

    dataset_results = []

    for x in range(100):

        initial = pauli_string_to_word(f"X{x}")

        for term in problem_terms:

            problem = pauli_string_to_word(term["pauli_string"])

            comm = qml.commutator(initial, problem, pauli=True)

            if len(comm) == 0:
                continue

            dataset_results.append({

                "initial_qubit": x,

                "problem_term": term,

                "commutator": comm,

            })

    O1[dataset_name] = dataset_results

    print("Non-zero commutators:", len(dataset_results))

In [ ]:
print(len(O1))
print(O1["BBB_Martins"][:5])

In [ ]:
for dataset in O1:
    print(dataset, len(O1[dataset]))

# Convert PauliString to Hamiltonian format

In [ ]:
# Convert PennyLane PauliSentence -> Hamiltonian dictionary

def pauli_sentence_to_terms(sentence, prefactor=1):

    terms = []

    for pw, coeff in sentence.items():

        pauli_string = " ".join(
            f"{op}{wire}"
            for wire, op in sorted(pw.items())
        )

        terms.append({

            "coefficient": prefactor * coeff,

            "pauli_string": pauli_string,

            "locality": len(pw)

        })

    return terms

In [ ]:
problem_term = problem_hamiltonians["BBB_Martins"]["pauli_terms"][0]

comm = qml.commutator(

    pauli_string_to_word("X0"),

    pauli_string_to_word(problem_term["pauli_string"]),

    pauli=True

)

converted = pauli_sentence_to_terms(

    comm,

    prefactor=problem_term["coefficient"]

)

print(converted)

# Build the first-order operator O_1

In [ ]:
# FIRST-ORDER OPERATOR

O1_hamiltonians = {}

for dataset_name, commutators in O1.items():

    pauli_terms = []

    for item in commutators:

        problem_coeff = item["problem_term"]["coefficient"]

        converted_terms = pauli_sentence_to_terms(
            item["commutator"],
            prefactor=problem_coeff
        )

        pauli_terms.extend(converted_terms)

    O1_hamiltonians[dataset_name] = {

        "hamiltonian_type": "O1",

        "formula": "O1 = [H_initial, H_problem]",

        "selected_features":
            problem_hamiltonians[dataset_name]["selected_features"],

        "X_selected_scaled":
            problem_hamiltonians[dataset_name]["X_selected_scaled"],

        "pauli_terms":
            pauli_terms
    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "Pauli terms"
    )

In [ ]:
print(len(O1_hamiltonians))

print(O1_hamiltonians["BBB_Martins"]["pauli_terms"][:5])

In [ ]:
for dataset in O1_hamiltonians:
    print(
        dataset,
        len(O1_hamiltonians[dataset]["pauli_terms"])
    )

# Merge H_{\mathrm{ad}}

In [ ]:
# BUILD FULL ADIABATIC HAMILTONIAN TERM LIST

Had_hamiltonians = {}

for dataset_name, Had in adiabatic_hamiltonians.items():

    pauli_terms = []

    pauli_terms.extend(
        deepcopy(Had["problem_hamiltonian"])
    )

    pauli_terms.extend(
        deepcopy(Had["initial_hamiltonian"])
    )

    Had_hamiltonians[dataset_name] = {

        "hamiltonian_type": "H_ad",

        "formula": Had["formula"],

        "selected_features":
            Had["selected_features"],

        "X_selected_scaled":
            Had["X_selected_scaled"],

        "pauli_terms":
            pauli_terms
    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "Pauli terms"
    )

# Build the commutators for O_2

In [ ]:
print(adiabatic_hamiltonians.keys())

dataset = next(iter(adiabatic_hamiltonians))

print(dataset)
print(adiabatic_hamiltonians[dataset].keys())

## First-Order Adiabatic Gauge Potential Construction

Up to this point, we have constructed the Hamiltonians required for the nested-commutator approximation of the Adiabatic Gauge Potential (AGP), which forms the basis of the counterdiabatic Hamiltonian.

### 1. Problem Hamiltonian

The problem Hamiltonian was constructed as

$begin:math:display$
H\_\{\\mathrm\{problem\}\} \= H\_A \+ H\_B\,
$end:math:display$

where:

- **Hamiltonian A** contains the feature-interaction (spin-glass/Ising) couplings.
- **Hamiltonian B** contains the single-feature bias terms.

This follows the Hamiltonian construction used in the PolarisQB workflow.

---

### 2. Initial Hamiltonian

The initial Hamiltonian consists of transverse-field X operators,

$begin:math:display$
H\_\{\\mathrm\{initial\}\} \= \\sum\_i X\_i\,
$end:math:display$

which prepares the system for adiabatic evolution.

---

### 3. Adiabatic Hamiltonian

The complete adiabatic Hamiltonian is

$begin:math:display$
H\_\{\\mathrm\{ad\}\} \= H\_\{\\mathrm\{problem\}\} \+ H\_\{\\mathrm\{initial\}\}\.
$end:math:display$

This is the Hamiltonian whose adiabatic evolution we ultimately wish to accelerate.

---

### 4. Zeroth Operator

The first operator in the nested-commutator expansion is

$begin:math:display$
O\_0\=\\frac\{\\partial H\_\{\\mathrm\{ad\}\}\}\{\\partial\\lambda\}\.
$end:math:display$

For the linear interpolation used here,

$begin:math:display$
O\_0 \= H\_\{\\mathrm\{problem\}\} \- H\_\{\\mathrm\{initial\}\}\.
$end:math:display$

---

### 5. First Nested Commutator

Using PennyLane's verified Pauli commutator implementation, we constructed

$begin:math:display$
O\_1\=\[H\_\{\\mathrm\{ad\}\}\,O\_0\]\,
$end:math:display$

which is represented in the same Hamiltonian dictionary format as the rest of the project.

The operator $begin:math:text$O\_1$end:math:text$ provides the first nontrivial contribution to the Adiabatic Gauge Potential.

---

The next step is to compute the second nested commutator,

$begin:math:display$
O\_2\=\[H\_\{\\mathrm\{ad\}\}\,O\_1\]\,
$end:math:display$

which is required for determining the variational coefficient $begin:math:text$\\alpha\_1$end:math:text$ used to construct the approximate Adiabatic Gauge Potential and, subsequently, the counterdiabatic Hamiltonian.

In [ ]:
# COMPUTE COMMUTATOR TERMS FOR O2
# O2 = [H_ad, O1]

import pennylane as qml

O2 = {}

for dataset_name in Had_hamiltonians:

    print(f"Processing {dataset_name}...")

    Had_terms = Had_hamiltonians[dataset_name]["pauli_terms"]

    O1_terms = O1_hamiltonians[dataset_name]["pauli_terms"]

    dataset_results = []

    for had_term in Had_terms:

        had = pauli_string_to_word(
            had_term["pauli_string"]
        )

        for o1_term in O1_terms:

            o1 = pauli_string_to_word(
                o1_term["pauli_string"]
            )

            comm = qml.commutator(
                had,
                o1,
                pauli=True
            )

            if len(comm) == 0:
                continue

            total_prefactor = (
                had_term["coefficient"]
                * o1_term["coefficient"]
            )

            dataset_results.append({

                "commutator": comm,

                "prefactor": total_prefactor,

                "had_term": had_term,

                "o1_term": o1_term

            })

    O2[dataset_name] = dataset_results

    print(
        "Non-zero commutators:",
        len(dataset_results)
    )

# Sanity check! 

In [ ]:
print(len(O2))

print(O2["BBB_Martins"][:5])

In [ ]:
for dataset in O2:
    print(dataset, len(O2[dataset]))

In [ ]:

# BUILD SECOND-ORDER OPERATOR O2

O2_hamiltonians = {}

for dataset_name, commutators in O2.items():

    pauli_terms = []

    for item in commutators:

        converted_terms = pauli_sentence_to_terms(
            item["commutator"],
            prefactor=item["prefactor"]
        )

        pauli_terms.extend(converted_terms)

    O2_hamiltonians[dataset_name] = {

        "hamiltonian_type": "O2",

        "formula": "O2 = [H_ad, O1]",

        "selected_features":
            Had_hamiltonians[dataset_name]["selected_features"],

        "X_selected_scaled":
            Had_hamiltonians[dataset_name]["X_selected_scaled"],

        "pauli_terms":
            pauli_terms
    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "Pauli terms"
    )

In [ ]:
print(len(O2_hamiltonians))

print(O2_hamiltonians["BBB_Martins"]["pauli_terms"][:5])

for dataset in O2_hamiltonians:
    print(dataset, len(O2_hamiltonians[dataset]["pauli_terms"]))

Inspect Duplication of Pauli terms for simplification 

In [ ]:
from collections import Counter

print("===== O2 Duplicate Inspection =====\n")

for dataset in O2_hamiltonians:

    strings = Counter(
        term["pauli_string"]
        for term in O2_hamiltonians[dataset]["pauli_terms"]
    )

    total = len(O2_hamiltonians[dataset]["pauli_terms"])
    unique = len(strings)
    duplicates = total - unique

    print(dataset)
    print(f"  Total terms      : {total}")
    print(f"  Unique strings   : {unique}")
    print(f"  Duplicate terms  : {duplicates}")
    print("-" * 50)

In [ ]:
dataset = "BBB_Martins"

strings = Counter(
    term["pauli_string"]
    for term in O2_hamiltonians[dataset]["pauli_terms"]
)

print("Most common Pauli strings:\n")

for pauli, count in strings.most_common(20):

    if count > 1:
        print(f"{pauli:20s} {count}")

In [ ]:
from collections import Counter

print("===== Locality Distribution =====\n")

for dataset in O2_hamiltonians:

    locality = Counter()

    for term in O2_hamiltonians[dataset]["pauli_terms"]:
        locality[term["locality"]] += 1

    print(dataset)
    print(locality)
    print("-" * 50)

In [ ]:
from collections import Counter

print("===== Pauli Letter Counts =====\n")

for dataset in O2_hamiltonians:

    letters = Counter()

    for term in O2_hamiltonians[dataset]["pauli_terms"]:

        for token in term["pauli_string"].split():

            letters[token[0]] += 1

    print(dataset)
    print(letters)
    print("-" * 50)

# Simplification

In [ ]:
# MERGE IDENTICAL PAULI TERMS IN O2

from collections import defaultdict

O2_compressed = {}

for dataset_name, H in O2_hamiltonians.items():

    merged = defaultdict(complex)

    locality = {}

    for term in H["pauli_terms"]:

        key = term["pauli_string"]

        merged[key] += term["coefficient"]

        locality[key] = term["locality"]

    pauli_terms = []

    for key, coeff in merged.items():

        pauli_terms.append({

            "coefficient": coeff,

            "pauli_string": key,

            "locality": locality[key]

        })

    O2_compressed[dataset_name] = {

        "hamiltonian_type": "O2_compressed",

        "formula": H["formula"],

        "selected_features": H["selected_features"],

        "X_selected_scaled": H["X_selected_scaled"],

        "pauli_terms": pauli_terms

    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "unique Pauli terms"
    )

In [ ]:
# Validation

from collections import Counter

print("===== O2 Compression Summary =====\n")

for dataset in O2_hamiltonians:

    raw = len(O2_hamiltonians[dataset]["pauli_terms"])
    compressed = len(O2_compressed[dataset]["pauli_terms"])

    print(dataset)
    print(f"Raw terms        : {raw}")
    print(f"Compressed terms : {compressed}")
    print(f"Reduction        : {100*(1-compressed/raw):.2f}%")
    print("-"*50)

## Variational Adiabatic Gauge Potential (AGP)

At this stage, the nested-commutator basis has been fully constructed.

### Completed Operator Construction

The following operators have been derived:

- **Problem Hamiltonian**

$begin:math:display$
H\_\{\\mathrm\{problem\}\} \= H\_A \+ H\_B
$end:math:display$

- **Initial Hamiltonian**

$begin:math:display$
H\_\{\\mathrm\{initial\}\} \= \\sum\_i X\_i
$end:math:display$

- **Adiabatic Hamiltonian**

$begin:math:display$
H\_\{\\mathrm\{ad\}\} \= H\_\{\\mathrm\{problem\}\} \+ H\_\{\\mathrm\{initial\}\}
$end:math:display$

- **Nested commutator basis**

$begin:math:display$
O\_0\=\\frac\{\\partial H\_\{\\mathrm\{ad\}\}\}\{\\partial\\lambda\}\,
$end:math:display$

$begin:math:display$
O\_1\=\[H\_\{\\mathrm\{ad\}\}\,O\_0\]\,
$end:math:display$

$begin:math:display$
O\_2\=\[H\_\{\\mathrm\{ad\}\}\,O\_1\]\.
$end:math:display$

These operators form the first-order Krylov basis used to approximate the Adiabatic Gauge Potential (AGP).

---

### Operator Simplification

The raw second-order operator $begin:math:text$O\_2$end:math:text$ contained many duplicate Pauli strings generated through different commutator paths.

Since a Hamiltonian is represented as a linear combination of Pauli operators,

$begin:math:display$
H\=\\sum\_i c\_iP\_i\,
$end:math:display$

identical Pauli strings were merged by summing their coefficients within each individual dataset. This is an **exact algebraic simplification** and does not alter the operator.

For the nine datasets containing 5,150 Hamiltonian terms:

- Raw $begin:math:text$O\_2$end:math:text$: **1,019,900** Pauli terms
- Canonical $begin:math:text$O\_2$end:math:text$: **505,100** unique Pauli terms
- Reduction: **50.48%**

The DILI dataset exhibited the same behaviour with proportionally fewer terms due to its smaller problem Hamiltonian.

---

### Next Step

The operator basis is now complete.

The next stage is the **variational optimization** of the Adiabatic Gauge Potential.

Rather than generating additional operators, the Frobenius norms of $begin:math:text$O\_1$end:math:text$ and $begin:math:text$O\_2$end:math:text$ will be computed to determine the first-order variational coefficient

$begin:math:display$
\\alpha\_1\,
$end:math:display$

from which the approximate AGP,

$begin:math:display$
A\_\\lambda\^\{\(1\)\} \= i\\alpha\_1O\_1\,
$end:math:display$

and the corresponding counterdiabatic Hamiltonian,

$begin:math:display$
H\_\{\\mathrm\{CD\}\}\=\\dot\{\\lambda\}A\_\\lambda\,
$end:math:display$

will be constructed. This final Hamiltonian will then be exported for block encoding and Hamiltonian simulation using the QSVT pipeline.

In [ ]:
# COMPUTE FROBENIUS NORMS OF O1 AND O2

frobenius_norms = {}

for dataset_name in O1_hamiltonians:

    o1_norm_sq = sum(
        abs(term["coefficient"])**2
        for term in O1_hamiltonians[dataset_name]["pauli_terms"]
    )

    o2_norm_sq = sum(
        abs(term["coefficient"])**2
        for term in O2_compressed[dataset_name]["pauli_terms"]
    )

    frobenius_norms[dataset_name] = {

        "O1_norm_sq": o1_norm_sq,
        "O2_norm_sq": o2_norm_sq

    }

    print(dataset_name)
    print(f"||O1||² = {o1_norm_sq:.6e}")
    print(f"||O2||² = {o2_norm_sq:.6e}")
    print("-" * 60)

# VARIATIONAL COEFFICIENT α1

In [ ]:
# COMPUTE FIRST-ORDER VARIATIONAL COEFFICIENT α1

alpha1 = {}

for dataset_name, norms in frobenius_norms.items():

    alpha = -(
        norms["O1_norm_sq"] /
        norms["O2_norm_sq"]
    )

    alpha1[dataset_name] = alpha

    print(dataset_name)
    print(f"α1 = {alpha:.10f}")
    print("-" * 60)

## First-Order Adiabatic Gauge Potential (AGP)

The nested-commutator basis has now been constructed and the first-order variational coefficient, $begin:math:text$\\alpha\_1$end:math:text$, has been determined.

The Adiabatic Gauge Potential (AGP) is the auxiliary operator that generates counterdiabatic evolution by suppressing non-adiabatic transitions during the Hamiltonian evolution. Rather than computing the exact AGP, which generally requires diagonalizing the full Hamiltonian and produces highly non-local operators, a first-order nested-commutator approximation is employed.

The operator basis is

$begin:math:display$

O\_0\=\\frac\{\\partial H\_\{\\mathrm\{ad\}\}\}\{\\partial\\lambda\}\,

$end:math:display$

$begin:math:display$

O\_1\=\[H\_\{\\mathrm\{ad\}\}\,O\_0\]\,

$end:math:display$

$begin:math:display$

O\_2\=\[H\_\{\\mathrm\{ad\}\}\,O\_1\]\.

$end:math:display$

The first-order variational ansatz is

$begin:math:display$

A\_\\lambda\^\{\(1\)\} \= i\\alpha\_1 O\_1\,

$end:math:display$

where the coefficient $begin:math:text$\\alpha\_1$end:math:text$ is obtained by minimizing the AGP action

$begin:math:display$

S\_\\lambda\=\\mathrm\{Tr\}\\\!\\left\(G\_\\lambda\^\\dagger G\_\\lambda\\right\)\,

$end:math:display$

resulting in

$begin:math:display$

\\alpha\_1

\=

\-

\\frac\{\\\|O\_1\\\|\_F\^2\}

\{\\\|O\_2\\\|\_F\^2\}\.

$end:math:display$

Using the optimized coefficient, the approximate AGP is constructed for each ADMET dataset by scaling every Pauli term in $begin:math:text$O\_1$end:math:text$ by $begin:math:text$i\\alpha\_1$end:math:text$.

The resulting AGP provides the counterdiabatic correction required to construct the complete counterdiabatic Hamiltonian,

$begin:math:display$

H\_\{\\mathrm\{CD\}\}

\=

H\_\{\\mathrm\{ad\}\}

\+

\\dot\{\\lambda\}A\_\\lambda\,

$end:math:display$

which will subsequently be block encoded, qubitized, and transformed using QSVT. This avoids explicit Hamiltonian diagonalization while remaining faithful to the variational nested-commutator formulation of counterdiabatic driving.  [oai_citation:0‡assets-eu.researchsquare.com](https://assets-eu.researchsquare.com/files/rs-6306195/v1/1d61f2aeacc7e26da4f0b3b1.pdf?utm_source=chatgpt.com)

In [ ]:
# BUILD FIRST-ORDER AGP
# A_lambda = i * alpha1 * O1

AGP_hamiltonians = {}

for dataset_name in O1_hamiltonians:

    agp_terms = []

    alpha = alpha1[dataset_name]

    for term in O1_hamiltonians[dataset_name]["pauli_terms"]:

        agp_terms.append({

            "coefficient": 1j * alpha * term["coefficient"],

            "pauli_string": term["pauli_string"],

            "locality": term["locality"]

        })

    AGP_hamiltonians[dataset_name] = {

        "hamiltonian_type": "AGP",

        "formula": "A_lambda = i * alpha1 * O1",

        "selected_features":
            O1_hamiltonians[dataset_name]["selected_features"],

        "X_selected_scaled":
            O1_hamiltonians[dataset_name]["X_selected_scaled"],

        "pauli_terms": agp_terms

    }

    print(
        dataset_name,
        "->",
        len(agp_terms),
        "Pauli terms"
    )

In [ ]:
print(len(AGP_hamiltonians))
print(AGP_hamiltonians["BBB_Martins"]["pauli_terms"][:5])

for dataset in AGP_hamiltonians:
    print(dataset, len(AGP_hamiltonians[dataset]["pauli_terms"]))

# Counter-daibatic Hamiltonian

The first-order Adiabatic Gauge Potential (AGP) has now been constructed for each dataset using the optimized variational coefficient,

$begin:math:display$
A\_\\lambda\^\{\(1\)\} \= i\\alpha\_1 O\_1\.
$end:math:display$

The corresponding counterdiabatic Hamiltonian is obtained by adding the AGP correction to the adiabatic Hamiltonian,

$begin:math:display$
H\_\{\\mathrm\{CD\}\}
\=
H\_\{\\mathrm\{ad\}\}
\+
\\dot\{\\lambda\}A\_\\lambda\,
$end:math:display$

where $begin:math:text$\\dot\{\\lambda\}$end:math:text$ denotes the derivative of the annealing schedule. The AGP introduces the additional driving terms required to suppress non-adiabatic transitions during the evolution without requiring explicit diagonalization of the Hamiltonian.  [oai_citation:0‡Nature](https://www.nature.com/articles/s42005-024-01807-2?utm_source=chatgpt.com)

The resulting $begin:math:text$H\_\{\\mathrm\{CD\}\}$end:math:text$ is the final Hamiltonian that will be used throughout the remainder of the workflow. In the next stage, this Hamiltonian will be decomposed into an LCU representation, block encoded, qubitized through the walk operator, and finally transformed using Quantum Singular Value Transformation (QSVT).

In [ ]:
# BUILD COUNTERDIABATIC HAMILTONIAN
# H_CD = H_ad + lambda_dot * AGP

lambda_dot = 1.0

HCD_hamiltonians = {}

for dataset_name in Had_hamiltonians:

    had_terms = list(Had_hamiltonians[dataset_name]["pauli_terms"])

    cd_terms = []

    for term in AGP_hamiltonians[dataset_name]["pauli_terms"]:

        new_term = term.copy()

        new_term["coefficient"] *= lambda_dot

        cd_terms.append(new_term)

    HCD_hamiltonians[dataset_name] = {

        "hamiltonian_type": "counterdiabatic",

        "formula": "H_CD = H_ad + lambda_dot * A_lambda",

        "selected_features":
            Had_hamiltonians[dataset_name]["selected_features"],

        "X_selected_scaled":
            Had_hamiltonians[dataset_name]["X_selected_scaled"],

        "pauli_terms":
            had_terms + cd_terms

    }

    print(
        dataset_name,
        "->",
        len(HCD_hamiltonians[dataset_name]["pauli_terms"]),
        "Pauli terms"
    )

# Sanity Check

In [ ]:

# H_CD DUPLICATE INSPECTION

from collections import Counter

print("===== H_CD Duplicate Inspection =====\n")

for dataset in HCD_hamiltonians:

    strings = Counter(
        term["pauli_string"]
        for term in HCD_hamiltonians[dataset]["pauli_terms"]
    )

    total = len(HCD_hamiltonians[dataset]["pauli_terms"])
    unique = len(strings)
    duplicates = total - unique

    print(dataset)
    print(f"Total terms      : {total}")
    print(f"Unique strings   : {unique}")
    print(f"Duplicate terms  : {duplicates}")
    print("-"*60)

In [ ]:

# H_CD LOCALITY DISTRIBUTION

from collections import Counter

print("===== H_CD Locality Distribution =====\n")

for dataset in HCD_hamiltonians:

    locality = Counter()

    for term in HCD_hamiltonians[dataset]["pauli_terms"]:

        k = len(term["pauli_string"].split())

        locality[k] += 1

    print(dataset)
    print(locality)
    print("-"*60)

In [ ]:

# H_CD PAULI LETTER COUNTS

from collections import Counter

print("===== H_CD Pauli Letter Counts =====\n")

for dataset in HCD_hamiltonians:

    letters = Counter()

    for term in HCD_hamiltonians[dataset]["pauli_terms"]:

        for token in term["pauli_string"].split():

            letters[token[0]] += 1

    print(dataset)
    print(letters)
    print("-"*60)

In [ ]:
# MOST COMMON DUPLICATE PAULI STRINGS

from collections import Counter

dataset = "BBB_Martins"

strings = Counter(
    term["pauli_string"]
    for term in HCD_hamiltonians[dataset]["pauli_terms"]
)

print("Most common Pauli strings:\n")

for pauli, count in strings.most_common(20):

    if count > 1:
        print(f"{pauli:20s} {count}")

## Resource characterization

In [ ]:

# H_CD RESOURCE CHARACTERIZATION

import numpy as np
from collections import Counter

resource_summary = {}

print("===== H_CD RESOURCE SUMMARY =====\n")

for dataset, H in HCD_hamiltonians.items():

    terms = H["pauli_terms"]

    num_terms = len(terms)

    num_system_qubits = len(H["selected_features"])

    num_index_qubits = int(np.ceil(np.log2(num_terms)))

    lambda_lcu = sum(
        abs(term["coefficient"])
        for term in terms
    )

    coeffs = np.array([
        abs(term["coefficient"])
        for term in terms
    ])

    locality = Counter()

    for term in terms:

        locality[
            len(term["pauli_string"].split())
        ] += 1

    pauli_letters = Counter()

    for term in terms:

        for token in term["pauli_string"].split():

            pauli_letters[token[0]] += 1

    resource_summary[dataset] = {

        "system_qubits": num_system_qubits,

        "index_qubits": num_index_qubits,

        "num_terms": num_terms,

        "lambda": lambda_lcu,

        "max_coefficient": coeffs.max(),

        "mean_coefficient": coeffs.mean(),

        "locality": dict(locality),

        "pauli_letters": dict(pauli_letters)

    }

    print(dataset)

    print("System qubits :", num_system_qubits)

    print("Index qubits  :", num_index_qubits)

    print("Pauli terms   :", num_terms)

    print("Lambda        :", lambda_lcu)

    print("Max coeff     :", coeffs.max())

    print("Mean coeff    :", coeffs.mean())

    print("Locality      :", locality)

    print("Letters       :", pauli_letters)

    print("-"*70)

# Export CD-Hamiltonian 

In [ ]:
# EXPORT FINAL COUNTERDIABATIC HAMILTONIANS

import pickle

with open(
    ARTIFACT_DIR / "HCD_hamiltonians.pkl",
    "wb"
) as f:

    pickle.dump(HCD_hamiltonians, f)


with open(
    ARTIFACT_DIR / "HCD_resource_summary.pkl",
    "wb"
) as f:

    pickle.dump(resource_summary, f)


print("Export complete.")

print("Saved:")

print(" - HCD_hamiltonians.pkl")

print(" - HCD_resource_summary.pkl")